# 11 · Sparse attention: sliding window, block selection (NSA), token selection (DSA)

**Needed for:** Native Sparse Attention (NSA, DeepSeek 2025) or DeepSeek Sparse Attention (DSA, V3.2), LLM track
step 9, and the compressed sparse attention in V4.1-Flash (step 10).

Dense causal attention costs $O(T^2)$. Sparse attention lets each query attend to a small subset of keys.
The designs differ in *how the subset is chosen*:

| Method | Each query sees | Selection cost |
|---|---|---|
| Sliding window | the last $W$ tokens | none, fixed pattern |
| NSA compression | one summary token per block of keys | cheap, attend to $T/b$ summaries |
| NSA selection | the top-$n$ key **blocks**, scored via compressed keys | reuses the compression scores |
| DSA (V3.2) | the top-$k$ key **tokens**, scored by a tiny "lightning indexer" | a small $O(T^2)$ indexer in FP8, much cheaper than attention |

The code here builds explicit boolean masks, which is simple to check but saves no compute. Real kernels skip
the masked-out blocks entirely. Every method must reduce to dense causal attention when its budget covers
everything, so that's the main test.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")


def masked_attention(q, k, v, keep):
    """q, k, v: (B, H, T, d); keep: bool, broadcastable to (B, H, T, T). True = may attend."""
    return F.scaled_dot_product_attention(q, k, v, attn_mask=keep)


def causal_mask(T):
    return torch.ones(T, T, dtype=torch.bool).tril()

## 1. Sliding-window attention

Query $t$ sees keys $t-W+1 \dots t$. Used alone (Mistral) or interleaved with global layers (Gemma, GPT-OSS,
DeepSeek-V4's SWA branch). Only the last $W$ keys ever need to stay in the cache (see notebook 12).

In [ ]:
def sliding_window_mask(T, window):
    i = torch.arange(T)[:, None]
    j = torch.arange(T)[None, :]
    return (j <= i) & (j > i - window)


def test_sliding_window(fn):
    m = fn(6, 3)
    assert m[5].tolist() == [False, False, False, True, True, True]
    assert m[0].tolist() == [True, False, False, False, False, False]
    assert torch.equal(fn(10, 10), causal_mask(10))       # window >= T is dense causal


test_sliding_window(sliding_window_mask)
print(sliding_window_mask(6, 3).int())

## 2. NSA: compression and block selection

Split keys into blocks of size $b$.
- **Compression branch:** summarize each block into one key and one value (the paper uses a small learned MLP;
  mean pooling here). A query may only see blocks that are *complete* by its position.
- **Selection branch:** score each block by the query's attention to the compressed key, keep the top-$n$ blocks,
  and attend to all tokens inside them. The query's own block is always kept so local context is never lost.
- **Window branch:** sliding window attention.

The three outputs are combined with learned sigmoid gates per head. Queries in a GQA group share one selection,
so a kernel can load each selected block once for the whole group.

In [ ]:
def compress_blocks(k, v, block):
    """Mean-pool keys/values into blocks. (B, H, T, d) -> (B, H, nb, d), T divisible by block."""
    B, H, T, d = k.shape
    return k.view(B, H, T // block, block, d).mean(3), v.view(B, H, T // block, block, d).mean(3)


def block_selection_mask(q, k_cmp, block, n_select):
    """Token-level keep mask from top-n block selection. q: (B, H, T, d), k_cmp: (B, H, nb, d)."""
    B, H, T, d = q.shape
    nb = k_cmp.shape[2]
    scores = q @ k_cmp.transpose(-2, -1) / math.sqrt(d)              # (B, H, T, nb)
    q_block = torch.arange(T) // block
    blk = torch.arange(nb)
    visible = blk[None, :] <= q_block[:, None]                        # block has started by position t
    scores = scores.masked_fill(~visible, float("-inf"))
    scores = scores.masked_fill(blk[None, :] == q_block[:, None], float("inf"))   # always keep own block
    top = scores.topk(min(n_select, nb), dim=-1).indices
    chosen = torch.zeros(B, H, T, nb, dtype=torch.bool).scatter_(-1, top, True) & visible
    keep = chosen.repeat_interleave(block, dim=-1)                    # block -> token granularity
    return keep & causal_mask(T)


def compressed_attention(q, k_cmp, v_cmp, block):
    """Attend to block summaries; only blocks fully in the past are visible."""
    T, nb = q.shape[2], k_cmp.shape[2]
    complete = (torch.arange(nb)[None, :] + 1) * block - 1 <= torch.arange(T)[:, None]   # block end <= t
    # early queries see no complete block: give them a zero output instead of NaN
    out = F.scaled_dot_product_attention(q, k_cmp, v_cmp, attn_mask=complete | ~complete.any(-1, keepdim=True))
    return out * complete.any(-1)[:, None].float()


B, H, T, d, block = 2, 4, 64, 16, 8
q, k, v = (torch.randn(B, H, T, d) for _ in range(3))
dense = masked_attention(q, k, v, causal_mask(T))

k_cmp, v_cmp = compress_blocks(k, v, block)
keep_all = block_selection_mask(q, k_cmp, block, n_select=T // block)
torch.testing.assert_close(masked_attention(q, k, v, keep_all), dense)
print("selecting every block == dense causal attention")

keep = block_selection_mask(q, k_cmp, block, n_select=2)
print(f"with 2 blocks of {block}: each query attends to at most {keep.sum(-1).max().item()} of {T} keys")

gates = torch.sigmoid(torch.randn(3, H))[:, None, :, None, None]          # per-branch, per-head gates
o_nsa = (gates[0] * compressed_attention(q, k_cmp, v_cmp, block)
         + gates[1] * masked_attention(q, k, v, keep)
         + gates[2] * masked_attention(q, k, v, sliding_window_mask(T, 16)))
print("NSA output:", o_nsa.shape, "| finite:", torch.isfinite(o_nsa).all().item())

## 3. DSA: a lightning indexer picks the top-k tokens (DeepSeek-V3.2)

A tiny extra module with few heads scores every (query, key) pair:
$$I_{t,s} = \sum_{j} w_{t,j}\ \text{ReLU}\big(q^I_{t,j} \cdot k^I_s\big)$$
It's still $O(T^2)$, but with a handful of small heads in FP8 it costs far less than the main attention. Each
query then runs full attention only on its top-$k$ keys (2048 in V3.2). The indexer is trained to match the
main attention's distribution (a KL loss) during a warm-up stage.

In [ ]:
class LightningIndexer(nn.Module):
    def __init__(self, d_model, n_heads=4, d_head=16):
        super().__init__()
        self.q = nn.Linear(d_model, n_heads * d_head, bias=False)
        self.k = nn.Linear(d_model, d_head, bias=False)             # one key shared by indexer heads
        self.w = nn.Linear(d_model, n_heads, bias=False)            # per-query head weights
        self.h, self.dh = n_heads, d_head

    def forward(self, x):
        B, T, _ = x.shape
        qi = self.q(x).view(B, T, self.h, self.dh)
        ki = self.k(x)
        scores = torch.einsum("bthd,bsd->bths", qi, ki).relu()
        return torch.einsum("bths,bth->bts", scores, self.w(x))     # (B, T, T)


def topk_token_mask(index_scores, topk):
    """index_scores: (B, T, T). Keep each query's top-k causal keys. Returns (B, 1, T, T) bool."""
    T = index_scores.shape[-1]
    causal = causal_mask(T)
    s = index_scores.masked_fill(~causal, float("-inf"))
    idx = s.topk(min(topk, T), dim=-1).indices
    keep = torch.zeros_like(s, dtype=torch.bool).scatter_(-1, idx, True) & causal
    return keep[:, None]


def test_topk_mask(fn):
    torch.manual_seed(0)
    scores = torch.randn(2, 12, 12)
    assert torch.equal(fn(scores, 12)[:, 0], causal_mask(12).expand(2, 12, 12))   # k >= T is dense causal
    m = fn(scores, 3)[:, 0]
    assert (m.sum(-1) == torch.arange(1, 13).clamp(max=3)).all()          # exactly min(t+1, k) keys
    assert not (m & ~causal_mask(12)).any()                                # never the future
    best = scores[0, 11].topk(3).indices.sort().values
    assert m[0, 11].nonzero().flatten().tolist() == best.tolist()


test_topk_mask(topk_token_mask)

x = torch.randn(B, T, 64)
indexer = LightningIndexer(64)
keep = topk_token_mask(indexer(x), topk=16)
out = masked_attention(q, k, v, keep)
print("DSA output:", out.shape, "| keys per query:", keep.sum(-1).max().item(), "of", T)
print(f"attention FLOPs at T=128k, k=2048: dense/sparse = {128_000 / 2048:.0f}x")

## Things to be able to explain

- Why must the selection be at *block* granularity for NSA to be fast on GPUs? (Contiguous blocks load as
  coalesced tiles and feed tensor cores; scattered single tokens don't.)
- How can the top-k choice be trained when top-k isn't differentiable? (The attention over chosen keys is
  differentiable; NSA scores come from the compression branch, which does get gradients; DSA trains its indexer
  with a separate KL loss to the dense attention distribution.)
- What does the sliding window branch protect against? (The model learning to rely only on local context
  through selection, and the selection missing nearby tokens.)

## Drill

In [ ]:
def my_sliding_window_mask(T, window):
    raise NotImplementedError


def my_topk_token_mask(index_scores, topk):
    raise NotImplementedError


drill(test_sliding_window, my_sliding_window_mask)
drill(test_topk_mask, my_topk_token_mask)